# Dataset v3: ablation по новым train-спикерам

Этот диагностический эксперимент отвечает на один вопрос: какой из новых
анонимизированных train-спикеров помогает или мешает качеству на неизменном
validation-спикере `spk06`.

Модель, LRW-pretrained признаки, balanced sampler, validation split и seeds не
меняются. Test split не загружается. Notebook использует уже готовые feature
caches, поэтому повторная обработка видео не нужна.


## 1. Подключить Google Drive и загрузить feature caches


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from collections import Counter
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
cache_dir = drive_root / 'controlled_lrw_frontend_bigru_dataset_v3'
output_dir = drive_root / 'controlled_lrw_dataset_v3_speaker_ablation'
output_dir.mkdir(parents=True, exist_ok=True)

train_cache_path = cache_dir / 'dataset_v3_train_lrw_resnet18_features.pt'
val_cache_path = cache_dir / 'dataset_v3_val_lrw_resnet18_features.pt'

if not train_cache_path.exists() or not val_cache_path.exists():
    raise FileNotFoundError(
        'Dataset v3 LRW feature caches were not found. Run the dataset v3 '
        'multiseed notebook first.'
    )

train_cache = load_torch_file(train_cache_path)
val_cache = load_torch_file(val_cache_path)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

vocab = sorted(set(train_cache['words']) | set(val_cache['words']))
# Keep the class order used by all previous experiments.
preferred_order = [
    'есть', 'когда', 'чтобы', 'просто', 'будет',
    'может', 'значит', 'время', 'потом', 'человек',
]
if set(preferred_order) == set(vocab):
    vocab = preferred_order
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}

assert train_cache['features'].shape == (1901, 24, 512)
assert val_cache['features'].shape == (105, 24, 512)
assert set(train_cache['speakers']).isdisjoint(set(val_cache['speakers']))

print('device:', device)
print('train cache:', train_cache['features'].shape)
print('validation cache:', val_cache['features'].shape)
print('train speakers:', sorted(set(train_cache['speakers'])))
print('validation speakers:', sorted(set(val_cache['speakers'])))
print('vocab:', vocab)
print('TEST SPLIT IS NOT LOADED.')


## 2. Задать варианты состава train и проверить их размеры


In [ ]:
BASE_SPEAKERS = ['spk03', 'spk04', 'spk05', 'spk11', 'spk12']
NEW_SPEAKERS = ['spk16', 'spk17', 'spk18', 'spk19']

VARIANTS = {
    'base_v2': BASE_SPEAKERS,
    'base_plus_spk16': BASE_SPEAKERS + ['spk16'],
    'base_plus_spk17': BASE_SPEAKERS + ['spk17'],
    'base_plus_spk18': BASE_SPEAKERS + ['spk18'],
    'base_plus_spk19': BASE_SPEAKERS + ['spk19'],
    'all_v3': BASE_SPEAKERS + NEW_SPEAKERS,
}


def subset_cache(cache, speakers):
    allowed = set(speakers)
    indices = [
        index for index, speaker in enumerate(cache['speakers'])
        if speaker in allowed
    ]
    return {
        'features': cache['features'][indices],
        'words': [cache['words'][index] for index in indices],
        'speakers': [cache['speakers'][index] for index in indices],
        'clip_paths': [cache['clip_paths'][index] for index in indices],
    }


count_rows = []
for variant, speakers in VARIANTS.items():
    subset = subset_cache(train_cache, speakers)
    count_rows.append({
        'variant': variant,
        'clips': len(subset['words']),
        'speakers': len(set(subset['speakers'])),
        'speaker_word_groups': len(set(zip(subset['speakers'], subset['words']))),
    })

variant_counts = pd.DataFrame(count_rows)
display(variant_counts)

speaker_word_counts = pd.crosstab(
    pd.Series(train_cache['speakers'], name='speaker'),
    pd.Series(train_cache['words'], name='word'),
).reindex(columns=vocab, fill_value=0)
display(speaker_word_counts)

assert variant_counts.set_index('variant').loc['base_v2', 'clips'] == 1085
assert variant_counts.set_index('variant').loc['all_v3', 'clips'] == 1901


## 3. BiGRU, balanced sampler и метрики


In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_loaders(cache, seed):
    group_keys = list(zip(cache['speakers'], cache['words']))
    group_counts = Counter(group_keys)
    weights = torch.tensor(
        [1.0 / group_counts[key] for key in group_keys], dtype=torch.double
    )
    sampler = WeightedRandomSampler(
        weights,
        num_samples=len(weights),
        replacement=True,
        generator=torch.Generator().manual_seed(seed),
    )
    train_loader = DataLoader(
        FeatureSequenceDataset(cache),
        batch_size=32,
        sampler=sampler,
        num_workers=0,
    )
    val_loader = DataLoader(
        FeatureSequenceDataset(val_cache),
        batch_size=32,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, val_loader, len(group_counts)


def classification_metrics(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1

    f1_scores = []
    recalls = []
    for class_id in range(num_classes):
        tp = int(matrix[class_id, class_id])
        support = int(matrix[class_id].sum())
        predicted = int(matrix[:, class_id].sum())
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        if support:
            recalls.append(recall)
            f1_scores.append(f1)

    return {
        'accuracy': float(np.trace(matrix) / matrix.sum()),
        'macro_f1': float(np.mean(f1_scores)),
        'balanced_accuracy': float(np.mean(recalls)),
        'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))),
    }


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total = 0
    preds = []
    labels = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            total_loss += criterion(outputs, batch_y).item() * batch_y.size(0)
            total += batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())
    metrics = classification_metrics(preds, labels, len(vocab))
    metrics['loss'] = total_loss / total
    return metrics


preview_model = BiGRUClassifier(len(vocab)).to(device)
print('trainable parameters:', count_parameters(preview_model))


## 4. Одинаковое обучение для каждого варианта и seed


In [ ]:
def run_one_seed(variant, speakers, seed, num_epochs=30, patience=7):
    subset = subset_cache(train_cache, speakers)
    set_reproducible_seed(seed)
    train_loader, val_loader, group_count = make_loaders(subset, seed)
    set_reproducible_seed(seed)

    model = BiGRUClassifier(len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )

    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    best_row = None
    epochs_without_improvement = 0
    start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * batch_y.size(0)
            train_correct += (outputs.argmax(dim=1) == batch_y).sum().item()
            train_total += batch_y.size(0)

        metrics = evaluate(model, val_loader, criterion)
        train_loss = train_loss_sum / train_total
        train_accuracy = train_correct / train_total
        improved = (
            metrics['macro_f1'] > best_macro_f1
            or (
                metrics['macro_f1'] == best_macro_f1
                and metrics['loss'] < best_val_loss
            )
        )

        if improved:
            best_macro_f1 = metrics['macro_f1']
            best_val_loss = metrics['loss']
            epochs_without_improvement = 0
            best_row = {
                'variant': variant,
                'seed': seed,
                'best_epoch': epoch,
                'train_clips': len(subset['words']),
                'train_speakers': len(set(subset['speakers'])),
                'speaker_word_groups': group_count,
                'train_accuracy_at_best': train_accuracy,
                'train_loss_at_best': train_loss,
                'validation_loss': metrics['loss'],
                'validation_accuracy': metrics['accuracy'],
                'validation_macro_f1': metrics['macro_f1'],
                'validation_balanced_accuracy': metrics['balanced_accuracy'],
                'predicted_classes': metrics['predicted_classes'],
                'trainable_parameters': count_parameters(model),
            }
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    best_row['epochs_completed'] = epoch
    best_row['training_seconds'] = time.perf_counter() - start
    print(
        f"{variant:>17} seed={seed:4d} | clips={len(subset['words']):4d} | "
        f"macro-F1={best_row['validation_macro_f1']:.4f} | "
        f"acc={best_row['validation_accuracy']:.4f} | "
        f"{best_row['training_seconds']:.1f}s"
    )
    return best_row


## 5. Запустить 18 коротких controlled runs


In [ ]:
SEEDS = [42, 123, 2026]
runs = []

for variant, speakers in VARIANTS.items():
    for seed in SEEDS:
        runs.append(run_one_seed(variant, speakers, seed))

runs_df = pd.DataFrame(runs)
runs_path = output_dir / 'dataset_v3_speaker_ablation_runs.csv'
runs_df.to_csv(runs_path, index=False)
display(runs_df)
print('saved:', runs_path)
print('TEST SPLIT WAS NOT USED.')


## 6. Сводка и decision gate


In [ ]:
summary = runs_df.groupby('variant', sort=False).agg(
    clips=('train_clips', 'first'),
    speakers=('train_speakers', 'first'),
    macro_f1_mean=('validation_macro_f1', 'mean'),
    macro_f1_std=('validation_macro_f1', 'std'),
    accuracy_mean=('validation_accuracy', 'mean'),
    balanced_accuracy_mean=('validation_balanced_accuracy', 'mean'),
).reset_index()

base_mean = float(
    summary.loc[summary['variant'] == 'base_v2', 'macro_f1_mean'].iloc[0]
)
summary['macro_f1_delta_vs_base'] = summary['macro_f1_mean'] - base_mean
display(summary)

pivot = runs_df.pivot(
    index='seed', columns='variant', values='validation_macro_f1'
)
paired_deltas = pivot.subtract(pivot['base_v2'], axis=0)
display(paired_deltas)

summary.to_csv(output_dir / 'dataset_v3_speaker_ablation_summary.csv', index=False)
paired_deltas.to_csv(output_dir / 'dataset_v3_speaker_ablation_paired_deltas.csv')

candidates = summary[summary['variant'].str.startswith('base_plus_')].copy()
best = candidates.sort_values('macro_f1_mean').iloc[-1]
best_variant = best['variant']
wins = int((paired_deltas[best_variant] > 0).sum())

print('Base v2 mean macro-F1:', round(base_mean, 4))
print('Best single addition:', best_variant)
print('Best mean macro-F1:', round(float(best['macro_f1_mean']), 4))
print('Delta vs base:', round(float(best['macro_f1_delta_vs_base']), 4))
print('Wins against base by seed:', f'{wins}/{len(SEEDS)}')

if best['macro_f1_delta_vs_base'] > 0 and wins >= 2:
    print('DECISION: keep this speaker as a candidate for dataset v3.')
    print('Next: test useful combinations of the helpful speakers only.')
else:
    print('DECISION: no new speaker improves validation reliably alone.')
    print('Next: inspect automatic quality reports and domain mismatch.')

print('Do not open test during this selection.')
print('TEST SPLIT WAS NOT USED.')
